In [1]:
import pandas as pd, numpy as np, networkx as nx
from scipy import stats
import community.community_louvain as community_louvain

# ---------- 1. Bangun graf & partisi (persis Community Detection) ----------
edges_df = pd.read_csv('network_edges.csv')
G_directed = nx.DiGraph()
for _, r in edges_df.iterrows():
    G_directed.add_edge(r['source'], r['target'], weight=r['weight'])
G_und = nx.Graph()
for u, v, d in G_directed.edges(data=True):
    w = d.get('weight', 1)
    if G_und.has_edge(u, v): G_und[u][v]['weight'] += w
    else: G_und.add_edge(u, v, weight=w)

# ---------- 2. Data tweet untuk porosity (persis notebook 3.3) ----------
df = pd.read_csv('Data_with_community3.csv', sep=';')
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M')
shift_date = pd.Timestamp('2025-08-27')
df['phase'] = df['date'].apply(lambda x: 'Pre-shift' if x < shift_date else 'Post-shift')

def porosity_test(community_lookup):
    """Cell 4 + Cell 14 logic, parametrized by partition."""
    recs = []
    for _, row in df.iterrows():
        source = str(row.get('username_clean', row['username'])).lower().strip().replace('@','')
        targets = []
        if pd.notna(row.get('in_reply_to_screen_name')): targets.append(str(row['in_reply_to_screen_name']).lower().strip().replace('@',''))
        if pd.notna(row.get('quoted_username')): targets.append(str(row['quoted_username']).lower().strip().replace('@',''))
        if pd.notna(row.get('mentions')):
            for m in str(row['mentions']).replace('@','').split():
                if m.strip(): targets.append(m.lower().strip())
        for t in targets:
            if source != t:
                sc, tc = community_lookup.get(source), community_lookup.get(t)
                if sc is not None and tc is not None:
                    recs.append((int(sc), int(tc), sc != tc, row['phase']))
    e = pd.DataFrame(recs, columns=['source_community','target_community','is_cross','phase'])
    # porosity per community per phase, min 5 edges
    rows = []
    for comm in e['source_community'].unique():
        for ph in ['Pre-shift','Post-shift']:
            sub = e[(e['source_community']==comm)&(e['phase']==ph)]
            if len(sub) < 5: continue
            rows.append((comm, ph, sub['is_cross'].mean()))
    p = pd.DataFrame(rows, columns=['community','phase','porosity'])
    pre = p[p['phase']=='Pre-shift'].set_index('community')
    post = p[p['phase']=='Post-shift'].set_index('community')
    both = pre.join(post, lsuffix='_pre', rsuffix='_post', how='inner')
    if len(both) < 10: return None
    w, pval = stats.wilcoxon(both['porosity_pre'], both['porosity_post'])
    inc = (both['porosity_post'] > both['porosity_pre']).mean()*100
    return {'n': len(both), 'mean_pre': both['porosity_pre'].mean(),
            'mean_post': both['porosity_post'].mean(), 'W': w, 'p': pval, 'pct_increased': inc}

# ---------- 3. GATE: seed 42 harus reproduce W=178, p=0.025, 0.141->0.176 ----------
part42 = community_louvain.best_partition(G_und, weight='weight', resolution=1.0, random_state=42)
lookup42 = {str(k).lower().strip().replace('@',''): v for k, v in part42.items()}
base = porosity_test(lookup42)
print("GATE seed 42:", {k: (round(v,4) if isinstance(v,float) else v) for k,v in base.items()})
print("  -> harus ~ n=38, mean_pre~0.141, mean_post~0.176, W~178, p~0.025")

GATE seed 42: {'n': 38, 'mean_pre': np.float64(0.1407), 'mean_post': np.float64(0.1764), 'W': np.float64(178.0), 'p': np.float64(0.0248), 'pct_increased': np.float64(60.5263)}
  -> harus ~ n=38, mean_pre~0.141, mean_post~0.176, W~178, p~0.025


In [2]:
# ---------- 4. Lintas-seed (jalankan HANYA jika GATE lolos) ----------
print(f"\n{'seed':<5}{'n':<5}{'pre':<8}{'post':<8}{'W':<7}{'p':<9}{'%inc':<6}{'dir+sig?'}")
results = []
for s in range(1, 21):
    part = community_louvain.best_partition(G_und, weight='weight', resolution=1.0, random_state=s)
    lk = {str(k).lower().strip().replace('@',''): v for k, v in part.items()}
    r = porosity_test(lk)
    if r is None: continue
    ok = (r['mean_post'] > r['mean_pre']) and (r['p'] < 0.05)
    results.append((s, r, ok))
    print(f"{s:<5}{r['n']:<5}{r['mean_pre']:<8.3f}{r['mean_post']:<8.3f}{r['W']:<7.0f}{r['p']:<9.4f}{r['pct_increased']:<6.1f}{'YES' if ok else 'no'}")

n_ok = sum(1 for _,_,ok in results if ok)
print(f"\n{n_ok}/{len(results)} seed: porosity naik pre->post DAN signifikan (p<0.05)")


seed n    pre     post    W      p        %inc  dir+sig?
1    38   0.141   0.179   181    0.0282   55.3  YES
2    36   0.150   0.180   186    0.0913   58.3  no
3    38   0.144   0.189   155    0.0148   60.5  YES
4    36   0.144   0.178   152    0.0362   55.6  YES
5    37   0.150   0.182   166    0.0408   59.5  YES
6    40   0.138   0.179   185    0.0201   60.0  YES
7    38   0.157   0.174   214    0.1534   60.5  no
8    39   0.142   0.170   223    0.0840   56.4  no
9    40   0.161   0.177   259    0.2450   52.5  no
10   37   0.143   0.188   152    0.0129   59.5  YES
11   42   0.135   0.162   222    0.0313   57.1  YES
12   38   0.152   0.165   245    0.3694   57.9  no
13   42   0.139   0.183   232    0.0446   54.8  YES
14   39   0.138   0.176   194    0.0475   61.5  YES
15   39   0.149   0.170   221    0.1236   56.4  no
16   40   0.141   0.175   194    0.0290   60.0  YES
17   40   0.151   0.178   236    0.1275   57.5  no
18   39   0.135   0.172   182    0.0294   59.0  YES
19   39   0.1